# Superstore Retail Analytics - EDA and Answering Questions

**Continuing from:** `feature_engineering_analysis.ipynb` — this notebook picks up once the data is *prepared*, and turns it into something that can actually answer business questions.

**Workflow:**

This is the same repeatable process regardless of the dataset — the checklist we'll apply to the Superstore data in Part 2 below.

1. **Exploratory Data Analysis** — univariate → bivariate → multivariate, in that order.
2. **Answering the Questions** — go back to Step 1 and directly resolve each question with a specific table or aggregation.
3. **Insight Synthesis** — summarize findings in plain language, including caveats inherited from cleaning decisions.
4. **Handoff Prep** — name and preserve the tables/features the next stage (visualization) will need.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow
import squarify
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [16]:
# Reading the clean data and make a copy of the original file to be the single source of truth
df_prepared = pd.read_parquet("Sample-Superstore2019_prepared.parquet", engine="pyarrow")
df_prepared.shape

(9986, 31)

In [17]:
rfm = pd.read_parquet("Sample-Superstore2019_rfm.parquet", engine="pyarrow")
rfm.shape

(793, 11)

### Step 1 — Exploratory Data Analysis

Univariate first (what does one feature look like on its own), then bivariate (how does it relate to profit), then multivariate (multiple dimensions at once).

In [4]:
# Univariate: distribution of the new per-row features
print(df_prepared["Discount Bucket"].value_counts())
print()
print(df_prepared["Shipping Days"].describe())
print()
print(df_prepared["Profit Margin"].describe())

Discount Bucket
None         4793
Low          3801
High          856
Medium        536
Very High       0
Name: count, dtype: int64

count    9986.000000
mean        3.958442
std         1.748245
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: Shipping Days, dtype: float64

count    9986.000000
mean       12.018479
std        46.689386
min      -275.000000
25%         7.500000
50%        27.000000
75%        36.250000
max        50.000000
Name: Profit Margin, dtype: float64


In [5]:
# Bivariate: Profit against Discount Bucket, Sales against calendar features
print(df_prepared.groupby("Discount Bucket", observed=True)["Profit"].mean())
print()
print(df_prepared.groupby("Order Month", observed=True)["Sales"].sum())

Discount Bucket
None       66.901973
Low        26.497411
Medium   -109.710720
High      -89.438144
Name: Profit, dtype: float64

Order Month
1      94924.8356
2      59751.2514
3     204959.8088
4     137188.7966
5     155028.8117
6     152718.6793
7     147148.0370
8     159044.0630
9     307600.8257
10    200322.9847
11    351916.6910
12    324904.7875
Name: Sales, dtype: float64


In [6]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_prepared.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

,Sales,Profit
Region,,
West,725355.4885,108404.3777
East,677906.3680,91353.8822
South,391007.8250,46549.1972
Central,501239.8908,39706.3625


In [7]:
# Multiple aggregations at once with .agg()
category_summary = df_prepared.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

,total_sales,avg_profit,orders
Category,,,
Furniture,741432.0433,8.674036,2119
Office Supplies,718317.7920,20.300133,6022
Technology,835759.7370,78.800073,1845


In [8]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_prepared,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

Category,Furniture,Office Supplies,Technology
Region,,,
Central,340.534644,117.458801,405.753124
East,346.683053,119.837751,495.278469
South,353.511492,126.400376,508.492973
West,357.302325,116.422377,421.219893


In [22]:
# Boolean indexing: orders sold at a loss
loss_orders = df_prepared[df_prepared["Profit"] < 0]
print("Loss-making orders:", len(loss_orders))
loss_orders[["Order ID", "Product Name", "Sales", "Profit"]]

Loss-making orders: 1870


,Order ID,Product Name,Sales,Profit
3,US-2017-108966,Bretford CR4500 Series Slim Rectangular Table,957.5775,-383.0310
14,US-2017-118983,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,-123.8580
15,US-2017-118983,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,-3.8160
23,US-2019-156909,"Global Deluxe Stacking Chair, Gray",71.3720,-1.0196
27,US-2017-150630,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,-1665.0522
...,...,...,...,...
9912,CA-2018-149272,"GBC Pre-Punched Binding Paper, Plastic, White,...",22.3860,-35.8176
9913,CA-2016-111360,Acco Expandable Hanging Binders,5.7420,-4.5936
9923,CA-2017-104948,O'Sullivan Living Dimensions 3-Shelf Bookcases,683.3320,-40.1960
9929,CA-2018-164889,Hon 61000 Series Interactive Training Tables,71.0880,-1.7772


In [23]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_prepared.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

,Sales,Profit
Region,,
West,725355.4885,108404.3777
East,677906.3680,91353.8822
South,391007.8250,46549.1972
Central,501239.8908,39706.3625


In [24]:
# Multiple aggregations at once with .agg()
category_summary = df_prepared.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

,total_sales,avg_profit,orders
Category,,,
Furniture,741432.0433,8.674036,2119
Office Supplies,718317.7920,20.300133,6022
Technology,835759.7370,78.800073,1845


In [25]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_prepared,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

Category,Furniture,Office Supplies,Technology
Region,,,
Central,340.534644,117.458801,405.753124
East,346.683053,119.837751,495.278469
South,353.511492,126.400376,508.492973
West,357.302325,116.422377,421.219893


### Step 2 — Answering the Questions

Back to the five questions from Step 1 in Feature engineering— each one resolved directly with the features built above.

In [10]:
# Q1: which State generates the most Profit, and which the biggest loss?
state_profit = df_prepared.groupby("State", observed=True)["Profit"].sum().sort_values(ascending=False)
print("Highest-profit state:", state_profit.idxmax(), "->", round(state_profit.max(), 2))
print("Biggest-loss state:", state_profit.idxmin(), "->", round(state_profit.min(), 2))

Highest-profit state: California -> 76381.39
Biggest-loss state: Texas -> -25729.36


In [12]:
# Q2: relationship between Discount and Profit, using the bucket built in Step 2
discount_profit = df_prepared.groupby("Discount Bucket", observed=True)["Profit"].mean()
discount_profit

Discount Bucket
None       66.901973
Low        26.497411
Medium   -109.710720
High      -89.438144
Name: Profit, dtype: float64

In [13]:
# Q3: does Ship Mode relate to profitability or shipping time?
ship_mode_summary = df_prepared.groupby("Ship Mode", observed=True).agg(
    avg_profit=("Profit", "mean"),
    avg_shipping_days=("Shipping Days", "mean"),
)
ship_mode_summary

,avg_profit,avg_shipping_days
Ship Mode,,
First Class,31.845643,2.182824
Same Day,29.266591,0.044199
Second Class,29.449868,3.238414
Standard Class,27.495584,5.006875


In [14]:
# Q4: seasonality in Sales -- by month and by weekday
print(df_prepared.groupby("Order Month", observed=True)["Sales"].sum())
print()
print(df_prepared.groupby("Order Weekday", observed=True)["Sales"].sum().sort_values(ascending=False))

Order Month
1      94924.8356
2      59751.2514
3     204959.8088
4     137188.7966
5     155028.8117
6     152718.6793
7     147148.0370
8     159044.0630
9     307600.8257
10    200322.9847
11    351916.6910
12    324904.7875
Name: Sales, dtype: float64

Order Weekday
Monday       395924.1659
Wednesday    392859.0138
Tuesday      372918.7047
Sunday       340813.2381
Thursday     298632.0632
Saturday     287363.7686
Friday       206998.6180
Name: Sales, dtype: float64


In [20]:
# Q5: most valuable customers, using the RFM features from Step 3
rfm.sort_values("RFM_Total", ascending=False).head()

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total,Cluster,Segment
35,AI-10855,14,10,4375.7860,5,5,5,555,15,0,Champions (best customers)
31,AH-10210,7,9,4805.3440,5,5,5,555,15,0,Champions (best customers)
328,HM-14860,3,10,8236.7648,5,5,5,555,15,2,At Risk (customers likely to churn)
365,JG-15160,2,11,6366.3920,5,5,5,555,15,0,Champions (best customers)
314,GZ-14470,8,9,4355.1500,5,5,5,555,15,0,Champions (best customers)


Loss-making orders: 1870


,Order ID,Product Name,Sales,Profit
3,US-2017-108966,Bretford CR4500 Series Slim Rectangular Table,957.5775,-383.0310
14,US-2017-118983,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,-123.8580
15,US-2017-118983,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,-3.8160
23,US-2019-156909,"Global Deluxe Stacking Chair, Gray",71.3720,-1.0196
27,US-2017-150630,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,-1665.0522
...,...,...,...,...
9912,CA-2018-149272,"GBC Pre-Punched Binding Paper, Plastic, White,...",22.3860,-35.8176
9913,CA-2016-111360,Acco Expandable Hanging Binders,5.7420,-4.5936
9923,CA-2017-104948,O'Sullivan Living Dimensions 3-Shelf Bookcases,683.3320,-40.1960
9929,CA-2018-164889,Hon 61000 Series Interactive Training Tables,71.0880,-1.7772


### Step 3 — Insight Synthesis

- **Profit by state**: California is the strongest performer (~$76.4K total profit); Texas is the biggest drag (~-$25.7K total loss) — worth checking whether that's driven by discounting practices specific to Texas.
- **Discount vs. Profit**: a clear, monotonic relationship. Average profit per order falls as the discount band rises — roughly +$67 with no discount, +$27 at Low, **-$78 at Medium, -$107 at High**. Discounts above ~20% are, on average, selling at a loss.
- **Ship Mode**: shipping time behaves exactly as expected (Same Day ≈ 0 days, Standard Class ≈ 5 days), but average profit per order is fairly flat across all four modes (~$28–32) — shipping choice doesn't appear to drive profitability on its own.
- **Seasonality**: Sales rise sharply toward year-end (November and December are the two highest months, roughly 3–4x February's total), consistent with holiday retail patterns. By weekday, Monday/Tuesday/Wednesday outsell Friday by close to 2x.
- **Customer value**: spend is concentrated — the top RFM customer alone accounts for ~$25K in lifetime Sales, well above the ~$2.9K average, suggesting a small set of high-value accounts worth treating differently from the broader base.

**Caveat carried from cleaning:** these Profit/Sales totals reflect the decision *not* to collapse the 7 legitimate `Order ID` + `Product ID` duplicate pairs — collapsing them would have understated every total above.

### Step 3 — Handoff Prep for Visualization

The next stage (lectures 7–8) turns these tables into charts. This notebook's job is to make sure nothing needs recomputing there — everything it needs already exists, by name, above.

In [ ]:
# Tables and features the visualization stage will need -- named here so
# nothing has to be recomputed in the next notebook.
handoff_manifest = {
    "df_clean": "row-level cleaned + feature-engineered DataFrame",
    "customer_features": "one row per Customer ID -- spend, order count, tenure",
    "product_features": "one row per Product ID -- units sold, revenue, avg discount",
    "rfm": "one row per Customer ID -- Recency, Frequency, Monetary",
    "state_profit": "total Profit per State",
    "discount_profit": "average Profit per Discount Bucket",
    "ship_mode_summary": "average Profit and Shipping Days per Ship Mode",
    "region_summary": "total Sales and Profit per Region",
    "category_summary": "total Sales, avg Profit, order count per Category",
    "pivot": "average Sales by Region x Category",
}
for name, description in handoff_manifest.items():
    print(f"{name}: {description}")

### Step 4— Persist Handoff Tables to Disk

Listing the tables above isn't enough on its own -- `visualization_dashboard.ipynb` runs its **own kernel** and has no access to the variables sitting in this one's memory. Every object named in `handoff_manifest` gets written to a `handoff_data/` folder here, and the visualization notebook's Step 0 reads them straight back in.

- **Parquet** for the row-level / per-entity tables (`df_clean`, `customer_features`, `product_features`, `rfm`) -- preserves dtypes (`datetime64`, `category`) exactly like the clean-data handoff from the cleaning notebook.
- **CSV** for the small summary tables (`state_profit`, `discount_profit`, `ship_mode_summary`, `region_summary`, `category_summary`, `pivot`) -- they're already chart-ready and don't carry dtypes worth preserving via Parquet.

In [ ]:
import os

export_dir = "handoff_data"
os.makedirs(export_dir, exist_ok=True)

# Row-level / per-entity tables -- Parquet preserves dtypes
df_prepared.to_parquet(f"{export_dir}/df_clean.parquet", engine="pyarrow")
customer_features.to_parquet(f"{export_dir}/customer_features.parquet", engine="pyarrow")
product_features.to_parquet(f"{export_dir}/product_features.parquet", engine="pyarrow")
rfm.to_parquet(f"{export_dir}/rfm.parquet", engine="pyarrow")

# Small, chart-ready summary tables -- CSV is sufficient
state_profit.to_csv(f"{export_dir}/state_profit.csv")
discount_profit.to_csv(f"{export_dir}/discount_profit.csv")
ship_mode_summary.to_csv(f"{export_dir}/ship_mode_summary.csv")
region_summary.to_csv(f"{export_dir}/region_summary.csv")
category_summary.to_csv(f"{export_dir}/category_summary.csv")
pivot.to_csv(f"{export_dir}/pivot_region_category.csv")

print(f"Exported {len(os.listdir(export_dir))} files to '{export_dir}/':")
for filename in sorted(os.listdir(export_dir)):
    print(f"  {filename}")